# TransE, TransH y TransR — Bellicum

Solo estos tres modelos y solo el grafo de Bellicum, el más grande de los tres (120 cláusulas, 631 enunciados). Sus partes **no están verificadas a mano**: en el
paso 5, «la parte correcta» es la que puso la extracción. Cinco pasos:

1. el grafo como tripletas `(h, r, t)`;
2. qué hace cada modelo;
3. entrenarlos y ver qué han aprendido;
4. ¿adivinan aristas que no han visto?;
5. la pregunta que nos importa: ¿adivinan la parte de un enunciado?, y por qué no.

In [2]:
import collections
import sys
import time
from pathlib import Path

import numpy as np
import torch
from IPython.display import Markdown, display

# link_prediction.py está en notebooks/, una carpeta por encima de este cuaderno.
for folder in (Path.cwd(), Path.cwd().parent, Path.cwd() / "notebooks"):
    if (folder / "link_prediction.py").exists():
        sys.path.insert(0, str(folder))
        break
import link_prediction as lp

# Configuración de cada modelo: la que eligió el ajuste de link_prediction.ipynb (sección 3), hecho
# en SteelVault. Aquí se usa tal cual, sin ajustarla a este contrato.
#   loss           "1vsall": cada tripleta verdadera contra todas las entidades a la vez
#                  "margin": contra 32 tripletas falsas hechas al azar (pérdida de margen, la de los artículos)
#   lr             tasa de aprendizaje: cuánto se mueven los vectores en cada corrección
#   epochs         cuántas veces se recorren todas las tripletas
#   embedding_dim  números por entidad; TransR usa la mitad en el espacio de cada relación
#   batch_size     tripletas por corrección
#   negatives      tripletas falsas por cada verdadera (solo con "margin")
# El optimizador es Adam en los tres.
CONFIG = {
    "TransE": {"loss": "1vsall", "lr": 0.03, "epochs": 100, "embedding_dim": 128, "batch_size": 256, "negatives": 32},
    "TransH": {"loss": "margin", "lr": 0.1, "epochs": 300, "embedding_dim": 32, "batch_size": 256, "negatives": 32},
    "TransR": {"loss": "margin", "lr": 0.03, "epochs": 100, "embedding_dim": 32, "batch_size": 256, "negatives": 32},
}
MODELS = tuple(CONFIG)
# Veces que se repite el paso 5 con repartos distintos. 1, como link_prediction.ipynb hizo con
# este contrato; con 3, el paso 5 tarda el triple.
REPEATS = 1

contract_name = "Bellicum"
contract = lp.load_contract(contract_name)
e2i, r2i = lp.entity_ids(contract)
i2e = {i: e for e, i in e2i.items()}
print("dispositivo:", lp.DEVICE, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")

dispositivo: cuda | Quadro RTX 5000


In [3]:
ALL_TYPES = ("is_part_of", "assigns_obligation_to", "grants_right_to", "defines", "uses", "references", "depends_on", "supersedes", "modifies")
print(f"contrato {contract_name}")
print(f"{len(contract.triples)} tripletas, {len(e2i)} entidades, {len(r2i)} tipos de arista presentes")
print("tipos de arista que no aparecen:", ", ".join(r for r in ALL_TYPES if r not in r2i) or "ninguno")
lp.table([{"tipo de arista": r, "tripletas": n} for r, n in collections.Counter(r for _, r, _ in contract.triples).most_common()])

contrato Bellicum
2867 tripletas, 1326 entidades, 8 tipos de arista presentes
tipos de arista que no aparecen: supersedes


| tipo de arista | tripletas |
|---|---|
| is_part_of | 1199 |
| uses | 741 |
| assigns_obligation_to | 432 |
| grants_right_to | 178 |
| defines | 104 |
| modifies | 101 |
| references | 60 |
| depends_on | 52 |

Una tripleta de cada tipo de arista, con el texto de cada nodo para poder leerlas:

In [4]:
def label(node):
    if node in contract.parties:
        return f"{node} ({contract.parties[node]['name']})"
    if node in contract.clauses:
        c = contract.clauses[node]
        return f"{node} (§{c['ref']} {c.get('heading') or ''})".strip()
    if node in contract.statements:
        return f"{node} «{lp.statement_text(contract.statements[node])[:55]}…»"
    return node


examples = [next(t for t in contract.triples if t[1] == r) for r in sorted(r2i)]
lp.table([{"h": label(h), "r": r, "t": label(t)} for h, r, t in examples])

| h | r | t |
|---|---|---|
| obligation-1 «which addition or removal shall be mutually agreed by t…» | assigns_obligation_to | party-1 (Miltenyi Biotec GmbH) |
| clause-107 (§18.1 ) | defines | term-102 |
| clause-2 (§1.1 Definitions) | depends_on | clause-116 (§20.3 Entire Agreement and Amendment) |
| right-1 «such Module may be amended from time to time by written…» | grants_right_to | party-2 (Bellicum Pharmaceuticals, Inc.) |
| clause-10 (§2.4 Reserved Rights) | is_part_of | clause-6 (§2 SUPPLY OF PRODUCT; ALLIANCE MANAGERS; JOINT STEERING COMMITTEE) |
| obligation-209 «Should any of the Delivered Miltenyi Products be damage…» | modifies | clause-34 (§5.2 Volume Limitations) |
| clause-2 (§1.1 Definitions) | references | clause-11 (§2.5 Subcontracting by Miltenyi) |
| clause-100 (§15.5 ) | uses | term-46 |

## 2. Qué hace cada modelo

Los tres dan a cada entidad un vector de números (32 o 128, según la configuración) y a cada
tipo de arista otro vector. Una tripleta es «verdadera» si sus vectores encajan de una cierta
forma; cada modelo define esa forma:

| Modelo | Una tripleta es verdadera si… | Puntuación (más alta = más creíble) |
|---|---|---|
| **TransE** | origen + relación ≈ destino | `−‖h + r − t‖` |
| **TransH** | lo mismo, pero después de proyectar origen y destino sobre un **plano propio de cada relación** | `−‖h⊥ + r − t⊥‖`, con `h⊥ = h − (wᵣ·h) wᵣ` |
| **TransR** | lo mismo, pero después de llevar origen y destino a **un espacio propio de cada relación** con una matriz | `−‖Mᵣh + r − Mᵣt‖` |

**El problema del muchos-a-uno.** Cuántas aristas de parte apuntan a cada parte en este contrato:

In [5]:
counts = collections.Counter((r, t) for _, r, t in contract.triples if r in lp.PARTY_RELATIONS)
lp.table([{"tipo de arista": r, "parte": label(p), "aristas": n} for (r, p), n in sorted(counts.items())])

| tipo de arista | parte | aristas |
|---|---|---|
| assigns_obligation_to | party-1 (Miltenyi Biotec GmbH) | 216 |
| assigns_obligation_to | party-2 (Bellicum Pharmaceuticals, Inc.) | 216 |
| grants_right_to | party-1 (Miltenyi Biotec GmbH) | 95 |
| grants_right_to | party-2 (Bellicum Pharmaceuticals, Inc.) | 83 |

Para cada parte, TransE quiere que todas las aristas que apuntan a ella cumplan `h + r ≈ parte`,
así que empuja a todos esos enunciados hacia el mismo punto, `parte − r`. Pero esos enunciados
también tienen que estar en sitios distintos para sus otras aristas (cada uno en su cláusula).
TransH y TransR nacieron para esto: solo exigen que coincidan **en la proyección** de esa
relación, y fuera de ella cada enunciado puede estar donde le haga falta.

## 3. Entrenar y ver qué han aprendido

**Entrenar** es mover los vectores para que las tripletas verdaderas puntúen más que las falsas.
Para cada tripleta conocida se le pregunta al modelo `(h, r, ?)`: se puntúan todas las entidades
como posible destino, y se corrigen los vectores para que la verdadera quede arriba. Lo mismo
con el origen, `(?, r, t)`. Se repite unas cien o trescientas veces sobre todas las tripletas
(las *épocas*).

In [6]:
models = {}
rows = []
for m in MODELS:
    start = time.time()
    models[m] = lp.train_kge(m, contract.triples, e2i, r2i, seed=0, **CONFIG[m])
    rows.append({"modelo": m, **CONFIG[m], "segundos": round(time.time() - start, 1)})
lp.table(rows, ["modelo", "epochs", "embedding_dim", "lr", "loss", "segundos"])

/home/sante/Documents/FGV/me/Contract-Knowledge-Graph/server/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


| modelo | epochs | embedding_dim | lr | loss | segundos |
|---|---|---|---|---|---|
| TransE | 100 | 128 | 0.030 | 1vsall | 11.900 |
| TransH | 300 | 32 | 0.100 | margin | 39.000 |
| TransR | 100 | 32 | 0.030 | margin | 22.100 |

**Cómo se lee un modelo.**

Se le hace una pregunta: ¿a quién obliga una obligación concreta? La celda toma la primera
obligación del contrato que obliga a una sola parte. El modelo puntúa todas las entidades y las
ordena; la posición de la respuesta correcta es su **rango**: 1 si sale la primera.

Las cinco primeras de cada modelo. El modelo ha visto la respuesta al entrenar: esto mide
memoria, no deducción.

In [7]:
example = next(sid for sid, r, _ in lp.party_queries(contract) if r == "assigns_obligation_to" and sid not in contract.twin and contract.statements[sid]["kind"] == "obligation")
answer = contract.party_edge(example)[1]
display(Markdown(f"Pregunta: `({example}, assigns_obligation_to, ?)`, «{lp.statement_text(contract.statements[example])[:100]}». Respuesta correcta: `{answer}` ({contract.parties[answer]['name']})."))


def top(model, h, r, k=5):
    scores = lp.kge_tail_scores(model, [(h, r)], e2i, r2i)[0]
    order = np.argsort(-scores)
    return [f"{i2e[i]} ({scores[i]:.2f})" for i in order[:k]]


lp.table([{"modelo": m, **{f"{k + 1}.º": x for k, x in enumerate(top(models[m], example, "assigns_obligation_to"))}} for m in MODELS])

Pregunta: `(obligation-3, assigns_obligation_to, ?)`, «Miltenyi agrees to supply Miltenyi Products to Bellicum under this Agreement, as specifically applic». Respuesta correcta: `party-1` (Miltenyi Biotec GmbH).

| modelo | 1.º | 2.º | 3.º | 4.º | 5.º |
|---|---|---|---|---|---|
| TransE | party-1 (-31.76) | party-2 (-32.70) | term-57 (-38.93) | condition-225 (-39.09) | term-24 (-39.09) |
| TransH | party-1 (-0.45) | party-2 (-0.61) | obligation-3 (-1.00) | condition-152 (-1.04) | obligation-232 (-1.05) |
| TransR | party-1 (-0.30) | party-2 (-0.44) | obligation-181 (-0.68) | obligation-228 (-0.85) | obligation-115 (-0.85) |

Tiene el formato Entidad (puntuación). Solo importa el orden: cada modelo usa su propia escala.

## 4. ¿Adivinan aristas que no han visto?

Ahora la prueba de verdad: se esconde un 10% de las tripletas al azar, se entrena con el resto y
se pregunta por las escondidas. Para resumir todos esos rangos en un número se usan dos medidas:

- **MRR** (*rango recíproco medio*): la media de `1 / rango`. Si la respuesta sale 1.ª, 2.ª y
  4.ª en tres preguntas, el MRR es `(1 + 1/2 + 1/4) / 3 = 0,58`. Vale 1 si siempre sale la primera.
- **Hits@k**: la fracción de preguntas en que la respuesta sale entre las *k* primeras.

El rango es **filtrado**: si una pregunta tiene varias respuestas verdaderas conocidas (una
cláusula tiene muchos enunciados), las otras no cuentan como error. Como referencia, el azar:
ordenar todas las entidades sin mirar nada.

In [8]:
rng = np.random.default_rng(0)
hidden_idx = set(rng.choice(len(contract.triples), size=len(contract.triples) // 10, replace=False).tolist())
train = [t for i, t in enumerate(contract.triples) if i not in hidden_idx]
hidden = [contract.triples[i] for i in sorted(hidden_idx)]
true_tails = collections.defaultdict(set)
for h, r, t in contract.triples:
    true_tails[(h, r)].add(e2i[t])


def filtered_ranks(model, triples):
    scores = lp.kge_tail_scores(model, [(h, r) for h, r, _ in triples], e2i, r2i)
    ranks = []
    for row, (h, r, t) in zip(scores, triples):
        row = row.copy()
        row[[i for i in true_tails[(h, r)] if i != e2i[t]]] = -np.inf
        ranks.append(1 + int(np.sum(row > row[e2i[t]])))
    return np.array(ranks)


n = len(e2i)
rows = [{"modelo": "al azar (esperado)", "MRR": float(np.mean([1 / k for k in range(1, n + 1)])), "Hits@1": 1 / n, "Hits@10": 10 / n}]
ranks_by_model = {}
for m in MODELS:
    ranks = filtered_ranks(lp.train_kge(m, train, e2i, r2i, seed=0, **CONFIG[m]), hidden)
    ranks_by_model[m] = ranks
    rows.append({"modelo": m, "MRR": float(np.mean(1 / ranks)), "Hits@1": float(np.mean(ranks == 1)), "Hits@10": float(np.mean(ranks <= 10))})
display(Markdown(f"{len(hidden)} tripletas escondidas de {len(contract.triples)}"))
lp.table(rows)

286 tripletas escondidas de 2867

| modelo | MRR | Hits@1 | Hits@10 |
|---|---|---|---|
| al azar (esperado) | 0.006 | 0.001 | 0.008 |
| TransE | 0.323 | 0.185 | 0.549 |
| TransH | 0.216 | 0.126 | 0.353 |
| TransR | 0.042 | 0.014 | 0.094 |

Las escondidas son muchas para mirarlas una a una. Primero, el MRR de cada modelo según el tipo
de arista escondida:

In [9]:
by_type = collections.defaultdict(list)
for k, (_, r, _) in enumerate(hidden):
    by_type[r].append(k)
lp.table([
    {"tipo de arista": r, "escondidas": len(ks), **{m: float(np.mean(1 / ranks_by_model[m][ks])) for m in MODELS}}
    for r, ks in sorted(by_type.items(), key=lambda item: -len(item[1]))
])

| tipo de arista | escondidas | TransE | TransH | TransR |
|---|---|---|---|---|
| is_part_of | 117 | 0.091 | 0.039 | 0.017 |
| uses | 73 | 0.265 | 0.087 | 0.072 |
| assigns_obligation_to | 46 | 0.739 | 0.743 | 0.006 |
| grants_right_to | 19 | 0.763 | 0.719 | 0.146 |
| modifies | 11 | 0.713 | 0.190 | 0.077 |
| defines | 8 | 0.200 | 0.003 | 0.055 |
| references | 6 | 0.518 | 0.105 | 0.003 |
| depends_on | 6 | 0.225 | 0.040 | 0.062 |

Y las quince primeras tripletas escondidas, con el rango de cada modelo:

In [10]:
lp.table([{"h": h, "r": r, "t": t, **{m: int(ranks_by_model[m][k]) for m in MODELS}} for k, (h, r, t) in enumerate(hidden[:15])])

| h | r | t | TransE | TransH | TransR |
|---|---|---|---|---|---|
| clause-105 | uses | term-3 | 10 | 6 | 41 |
| clause-11 | is_part_of | clause-6 | 9 | 211 | 236 |
| clause-110 | uses | term-6 | 1 | 17 | 1036 |
| clause-12 | uses | term-34 | 2 | 6 | 2 |
| clause-12 | uses | term-39 | 2 | 54 | 4 |
| clause-12 | uses | term-47 | 4 | 29 | 8 |
| clause-13 | uses | term-7 | 26 | 14 | 34 |
| clause-15 | uses | term-47 | 7 | 21 | 7 |
| clause-15 | uses | term-56 | 1 | 17 | 10 |
| clause-16 | uses | term-56 | 1 | 15 | 28 |
| clause-19 | is_part_of | clause-18 | 79 | 322 | 15 |
| clause-2 | defines | term-25 | 10 | 492 | 4 |
| clause-2 | defines | term-32 | 8 | 323 | 8 |
| clause-2 | defines | term-48 | 3 | 180 | 58 |
| clause-2 | defines | term-5 | 1 | 826 | 164 |

Cómo leerlas: un 1 es un acierto a la primera; un número alto, que la respuesta quedó muy abajo
entre todas las entidades. Fíjate en qué tipos de arista (`r`) dan rangos bajos y en cuáles altos,
y en si el origen `h` tiene otras aristas que el modelo haya podido ver.

TransH y TransR tienen más piezas que ajustar que TransE (un plano o una matriz por tipo de
arista) y los mismos datos: en un grafo con casi todos los nodos de una a tres aristas, les
faltan ejemplos para ajustarlas. No es un fallo del entrenamiento: con el mismo código, en
*Nations* —un grafo de prueba pequeño pero denso— TransH empata con TransE
([`link_prediction.ipynb`](../link_prediction.ipynb), sección 3).

## 5. La pregunta que nos importa: ¿qué parte?

Se esconde la arista de parte de un enunciado y se le pregunta al modelo cuál de las **dos**
partes encaja mejor. Con dos candidatas, el MRR no aporta nada que no diga el acierto (vale
`(1 + acierto) / 2`), así que se mide el **acierto**: la fracción de enunciados con la parte bien.

- Solo los **enunciados unilaterales** (la celda dice cuántos). Los gemelos —la misma frase
  emitida una vez por parte, como en *«each party shall…»*— tienen las dos partes por
  construcción y no hay nada que adivinar.
- Se reparten los unilaterales en cinco grupos; se esconde un grupo, se entrena con el resto del
  grafo y se pregunta por ese grupo. Así hasta pasar por los cinco, y todo `REPEATS` veces con
  repartos distintos.
- Dos referencias sin modelo: **siempre la parte más frecuente** y **la mayoría de su cláusula**
  (la parte que más aparece entre los otros enunciados de la misma cláusula).
- La parte «correcta» es la que puso la extracción: en este contrato nadie la ha revisado a mano.

In [11]:
queries = [q for q in lp.party_queries(contract) if q[0] not in contract.twin]
twins = [q for q in lp.party_queries(contract) if q[0] in contract.twin]
print(f"{len(queries)} enunciados unilaterales, {len(twins)} gemelos")
parties = sorted(contract.parties)
correct = collections.defaultdict(list)  # método -> [acierto de cada repetición]
predictions = collections.defaultdict(dict)  # método -> {enunciado: parte} de la primera repetición
start = time.time()
for rep in range(REPEATS):
    fold_of = lp.folds(len(queries), 5, seed=rep)
    hits = collections.Counter()
    for fold in range(5):
        test = [q for q, f in zip(queries, fold_of) if f == fold]
        known = [q for q, f in zip(queries, fold_of) if f != fold] + twins
        graph = [t for t in contract.triples if t not in set(test)]
        base = lp.t1_baselines(contract, known, test)
        guesses = {"siempre la más frecuente": base["mayoría"], "mayoría de su cláusula": base["cláusula"]}
        for m in MODELS:
            scores = lp.kge_tail_scores(lp.train_kge(m, graph, e2i, r2i, seed=rep, **CONFIG[m]), [(h, r) for h, r, _ in test], e2i, r2i)
            guesses[m] = {sid: parties[int(np.argmax([row[e2i[p]] for p in parties]))] for (sid, _, _), row in zip(test, scores)}
        for method, guess in guesses.items():
            hits[method] += sum(guess[sid] == p for sid, _, p in test)
            if rep == 0:
                predictions[method].update(guess)
    for method, k in hits.items():
        correct[method].append(k / len(queries))
print(f"{REPEATS * 5 * len(MODELS)} entrenamientos en {time.time() - start:.0f} s")
lp.table(sorted([{"método": m, "acierto": float(np.mean(v)), "peor repetición": min(v), "mejor repetición": max(v)} for m, v in correct.items()], key=lambda r: -r["acierto"]))

370 enunciados unilaterales, 240 gemelos
15 entrenamientos en 291 s


| método | acierto | peor repetición | mejor repetición |
|---|---|---|---|
| mayoría de su cláusula | 0.651 | 0.651 | 0.651 |
| TransE | 0.635 | 0.635 | 0.635 |
| TransH | 0.532 | 0.532 | 0.532 |
| TransR | 0.514 | 0.514 | 0.514 |
| siempre la más frecuente | 0.476 | 0.476 | 0.476 |

En [`link_prediction.ipynb`](../link_prediction.ipynb), con estos mismos modelos y el mismo reparto,
ninguno de los tres llegó a la mayoría de su cláusula (65%): TransE 64%, TransH 53%, TransR 51%.
Bellicum reparte sus obligaciones casi a medias entre las dos partes, así que «siempre la más
frecuente» se queda en 48%, por debajo de una moneda, y aquí los tres la superan. Esta tabla
debería repetirlo.

### Por qué

Al esconder la arista de parte, ¿qué le queda al modelo para decidir? Las otras aristas del
enunciado. Cuántas tiene cada uno de los unilaterales:

In [12]:
others = collections.Counter(
    sum(1 for h, r, t in contract.triples if sid in (h, t) and r not in lp.PARTY_RELATIONS) for sid, _, _ in queries
)
lp.table([{"aristas además de la de parte": k, "enunciados": others[k]} for k in sorted(others)])

| aristas además de la de parte | enunciados |
|---|---|
| 1 | 80 |
| 2 | 119 |
| 3 | 68 |
| 4 | 46 |
| 5 | 26 |
| 6 | 16 |
| 7 | 4 |
| 8 | 6 |
| 9 | 1 |
| 10 | 2 |
| 12 | 1 |
| 13 | 1 |

Esas aristas son casi siempre `is_part_of` hacia su cláusula y, a veces, un término definido que
usa, una condición o un valor que cuelgan de él. **Ninguna dice quién carga con el enunciado.** Lo
único que el modelo puede deducir es «se parece a los otros enunciados de su cláusula», que es
justo la mayoría de la cláusula. Los que no tienen ninguna otra arista quedan aislados: el modelo
no tiene nada que mirar.

Lo que dice quién carga está **en el texto**, no en el grafo. Unos casos en que TransE falla:

In [13]:
fails = [(sid, p) for sid, _, p in queries if predictions["TransE"][sid] != p][:6]
lp.table([{
    "enunciado": sid,
    "texto": lp.statement_text(contract.statements[sid])[:80],
    "lo que ve el modelo": ", ".join(f"{r} {t if h == sid else h}" for h, r, t in contract.triples if sid in (h, t) and r not in lp.PARTY_RELATIONS) or "nada",
    "TransE dice": contract.parties[predictions["TransE"][sid]]["name"][:22],
    "correcta": contract.parties[p]["name"][:22],
} for sid, p in fails])

| enunciado | texto | lo que ve el modelo | TransE dice | correcta |
|---|---|---|---|---|
| obligation-12 | During the Term of this Agreement, and subject to the terms and conditions hereo | is_part_of clause-7 | Bellicum Pharmaceutica | Miltenyi Biotec GmbH |
| obligation-13 | Bellicum or its Licensees or Subcontractors will purchase from Miltenyi, Milteny | is_part_of clause-7 | Miltenyi Biotec GmbH | Bellicum Pharmaceutica |
| obligation-26 | Bellicum shall inform Miltenyi in writing at least twelve (12) months in advance | is_part_of condition-13, is_part_of clause-9, is_part_of value-7 | Miltenyi Biotec GmbH | Bellicum Pharmaceutica |
| obligation-31 | Miltenyi shall have sole responsibility for ensuring, and shall ensure, that Mil | is_part_of clause-12 | Bellicum Pharmaceutica | Miltenyi Biotec GmbH |
| obligation-32 | be the sole responsibility of Miltenyi to obtain and maintain, and Miltenyi shal | is_part_of clause-12, uses term-20 | Bellicum Pharmaceutica | Miltenyi Biotec GmbH |
| obligation-35 | Miltenyi shall comply with all reasonable requests for assistance by Bellicum in | is_part_of condition-14, is_part_of clause-12 | Bellicum Pharmaceutica | Miltenyi Biotec GmbH |

## En resumen

- El grafo de Bellicum son las tripletas `(h, r, t)` del paso 1.
- El paso 4 dice cuánto aprenden los tres del grafo en general, y en qué tipos de arista.
- El paso 5 dice si adivinan la parte de un enunciado mejor que mirar su cláusula. Si no lo hacen,
  la razón es la del apartado *Por qué*: las aristas que le quedan a un enunciado no dicen quién
  carga con él.
- La configuración se ajustó en SteelVault y aquí se usa tal cual.